<a href="https://colab.research.google.com/github/natoeiei/128-356-Big-Data/blob/main/%E0%B8%81%E0%B8%A5%E0%B8%B8%E0%B9%88%E0%B8%A1%E0%B8%81%E0%B9%89%E0%B8%AD%E0%B8%99_LAB_Olist_E_Commerce.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

สมาชิกกลุ่มก้อน

นาย ชัยวัฒน์ ทองธาระ 6604820001

นางสาว ทมิตา กีรติโชติ 6604820002


# 13. 🧪 Lab Test: Olist E-Commerce Analytics (Real World Data)

**Dataset:** Brazilian E-Commerce Public Dataset by Olist  
**Source:** [Kaggle / GitHub](https://github.com/ayushic2899/Brazilian-E-Commerce-Public-Dataset-by-Olist)  
**Goal:** วิเคราะห์ยอดขายและพฤติกรรมลูกค้าจากข้อมูลจริง 3 ตาราง (Orders, Items, Products)

### 📥 Step 0: Download Data


In [29]:
!curl -L -o ./brazilian-ecommerce.zip\
  https://www.kaggle.com/api/v1/datasets/download/olistbr/brazilian-ecommerce

print("✅ Download Completed")


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 42.6M  100 42.6M    0     0  87.5M      0 --:--:-- --:--:-- --:--:-- 87.5M
✅ Download Completed


In [30]:

!unzip /content/brazilian-ecommerce.zip

Archive:  /content/brazilian-ecommerce.zip
  inflating: olist_customers_dataset.csv  
  inflating: olist_geolocation_dataset.csv  
  inflating: olist_order_items_dataset.csv  
  inflating: olist_order_payments_dataset.csv  
  inflating: olist_order_reviews_dataset.csv  
  inflating: olist_orders_dataset.csv  
  inflating: olist_products_dataset.csv  
  inflating: olist_sellers_dataset.csv  
  inflating: product_category_name_translation.csv  


In [37]:
!ls


archive.zip			  olist_order_reviews_dataset.csv
brazilian-ecommerce.zip		  olist_orders.csv
olist_customers.csv		  olist_orders_dataset.csv
olist_customers_dataset.csv	  olist_products.csv
olist_geolocation_dataset.csv	  olist_products_dataset.csv
olist_items.csv			  olist_sellers_dataset.csv
olist_order_items_dataset.csv	  product_category_name_translation.csv
olist_order_payments_dataset.csv  sample_data


In [31]:
!pip install pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Olist E-Commerce Analytics") \
    .getOrCreate()

print("✅ Spark Ready")


✅ Spark Ready


### 🛠️ Task 1: Load & Clean Data

1. อ่านไฟล์ CSV ทั้ง 4 ไฟล์เข้าสู่ Spark DataFrame
2. ตรวจสอบ Schema และจัดการ Type (ถ้าจำเป็น)
3. สร้าง Temp View เพื่อเตรียมพร้อมสำหรับ SQL


In [38]:
orders = spark.read.option("header", True).option("inferSchema", True) \
    .csv("olist_orders_dataset.csv")

items = spark.read.option("header", True).option("inferSchema", True) \
    .csv("olist_order_items_dataset.csv")

products = spark.read.option("header", True).option("inferSchema", True) \
    .csv("olist_products_dataset.csv")

customers = spark.read.option("header", True).option("inferSchema", True) \
    .csv("olist_customers_dataset.csv")


In [40]:
orders.printSchema()
items.printSchema()
products.printSchema()
customers.printSchema()


root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)

root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)

root
 |-- product_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_lenght: integer (nullable = true)
 |-- product_description_lenght: integer (nullable = true)
 |-- product_photos_qty: integer (nullable 

In [41]:
from pyspark.sql.functions import to_timestamp

orders = orders.withColumn(
    "order_purchase_timestamp",
    to_timestamp("order_purchase_timestamp")
)


In [42]:
orders.createOrReplaceTempView("orders")
items.createOrReplaceTempView("items")
products.createOrReplaceTempView("products")
customers.createOrReplaceTempView("customers")

print("✅ Temp Views Created")


✅ Temp Views Created


### 🔗 Task 2: Join Data

จงสร้าง `df_master` โดยการ Join ตารางดังนี้:
1. `orders` JOIN `items` (ON order_id)
2. JOIN `products` (ON product_id)
3. JOIN `customers` (ON customer_id)

> **Tip:** ตรวจสอบจำนวนแถวหลัง Join ว่าเพิ่มขึ้นหรือลดลงผิดปกติหรือไม่


In [43]:
df_master = orders \
    .join(items, "order_id") \
    .join(products, "product_id") \
    .join(customers, "customer_id")


In [44]:
print("Orders:", orders.count())
print("Items:", items.count())
print("Master:", df_master.count())


Orders: 99441
Items: 112650
Master: 112650


In [45]:
df_master.createOrReplaceTempView("df_master")


### 📊 Task 3: Analytics

ตอบคำถามต่อไปนี้ (เลือกใช้ SQL หรือ PySpark ก็ได้):

1. **Top Products:** สินค้าหมวดไหน (`product_category_name`) มียอดขายรวม (price) สูงที่สุด 5 อันดับแรก?
2. **Top Cities:** เมืองไหน (`customer_city`) มีจำนวนคำสั่งซื้อมากที่สุด 5 อันดับแรก?
3. **(Optional) Monthly Sales:** ยอดขายรวมแต่ละเดือนเป็นอย่างไร? (แนวโน้ม)


In [46]:
spark.sql("""
SELECT
    product_category_name,
    SUM(price) AS total_sales
FROM df_master
GROUP BY product_category_name
ORDER BY total_sales DESC
LIMIT 5
""").show()


+---------------------+------------------+
|product_category_name|       total_sales|
+---------------------+------------------+
|         beleza_saude| 1258681.340000017|
|   relogios_presentes|1205005.6800000127|
|      cama_mesa_banho|1036988.6800000388|
|        esporte_lazer| 988048.9700000194|
| informatica_acess...| 911954.3200000152|
+---------------------+------------------+



In [47]:
spark.sql("""
SELECT
    customer_city,
    COUNT(DISTINCT order_id) AS total_orders
FROM df_master
GROUP BY customer_city
ORDER BY total_orders DESC
LIMIT 5
""").show()


+--------------+------------+
| customer_city|total_orders|
+--------------+------------+
|     sao paulo|       15402|
|rio de janeiro|        6834|
|belo horizonte|        2750|
|      brasilia|        2116|
|      curitiba|        1510|
+--------------+------------+



In [48]:
from pyspark.sql.functions import date_format

df_month = df_master.withColumn(
    "month",
    date_format("order_purchase_timestamp", "yyyy-MM")
)

df_month.createOrReplaceTempView("monthly")

spark.sql("""
SELECT
    month,
    SUM(price) AS total_sales
FROM monthly
GROUP BY month
ORDER BY month
""").show()


+-------+------------------+
|  month|       total_sales|
+-------+------------------+
|2016-09|            267.36|
|2016-10| 49507.66000000009|
|2016-12|              10.9|
|2017-01|120312.87000000032|
|2017-02| 247303.0199999985|
|2017-03|374344.29999999516|
|2017-04| 359927.2299999956|
|2017-05| 506071.1399999948|
|2017-06| 433038.5999999939|
|2017-07| 498031.4799999967|
|2017-08| 573971.6799999981|
|2017-09| 624401.6899999977|
|2017-10| 664219.4299999973|
|2017-11|1010271.3700000155|
|2017-12| 743914.1700000004|
|2018-01|  950030.360000015|
|2018-02| 844178.7100000086|
|2018-03| 983213.4400000148|
|2018-04| 996647.7500000133|
|2018-05| 996517.6800000133|
+-------+------------------+
only showing top 20 rows
